# 00 · Colab runner: open this every session and click Runtime → Run all

It runs the next unfinished experiments from `configs/pipeline.yaml` and stops before the free-tier
GPU budget runs out. Everything is saved to Google Drive, so:

* if Colab disconnects, just open this notebook again and **Run all**: finished steps are skipped and
  training resumes from its last epoch;
* you don't need to watch it. Logs land in the `logs/` folder on Drive.

**First:** Runtime → Change runtime type → **T4 GPU**.

**Team members:** set `TEAM` in the first cell to the name you were given (`'team1'`, `'team2'` or `'team3'`).
You need two shortcuts in your *My Drive*: `govi-ai` (shared with you, view only) and `govi-ai-teamN`
(shared with you, can edit). Your results are written to `govi-ai-teamN`.


In [ ]:
# ======= the only line to change =======
TEAM = ''          # owner: '' (all steps) or 'main' (owner's share while team members run theirs)
                   # team members: 'team1', 'team2' or 'team3'
# ========================================
from google.colab import drive
import os
drive.mount('/content/drive')
MY = '/content/drive/MyDrive'
INPUTS = f'{MY}/govi-ai'                                   # dataset, features, audit (owner's folder)
GOVI = f'{MY}/govi-ai-{TEAM}' if TEAM.startswith('team') else INPUTS   # where results are written
DATA = '/content/data'
SESSION_HOURS = 3.5   # free tier: keep under the daily GPU allowance
for d, what in ((INPUTS, 'govi-ai'), (GOVI, os.path.basename(GOVI))):
    assert os.path.isdir(d), f'"{what}" not found in My Drive: open "Shared with me", right-click it, Organize > Add shortcut > My Drive'
print('inputs :', INPUTS, '
results:', GOVI)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: only CPU steps will run'


In [ ]:
!git clone -q https://github.com/Kalana-Lakshan/govi-isuru-ai-service.git /content/govi-isuru-ai-service 2>/dev/null || git -C /content/govi-isuru-ai-service pull -q
%cd /content/govi-isuru-ai-service
!pip install -q -e ".[train]"
!git log --oneline -1

In [ ]:
if not os.path.exists(f'{DATA}/manifest.csv'):   # local disk is wiped every session
    !python scripts/pack_dataset.py --unpack --out "$INPUTS/dataset" --data-root $DATA
TEAM_ARG = f'--team {TEAM}' if TEAM else ''
!python scripts/pipeline.py --govi "$GOVI" --inputs "$INPUTS" $TEAM_ARG --status


In [ ]:
!python scripts/pipeline.py --data-root $DATA --govi "$GOVI" --inputs "$INPUTS" $TEAM_ARG --hours $SESSION_HOURS


In [ ]:
!python scripts/pipeline.py --govi "$GOVI" --inputs "$INPUTS" $TEAM_ARG --status
